In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Importing dataset
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
# Handling categorical data
dataset = pd.get_dummies(dataset,drop_first=True,dtype=int)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [4]:
# Dropping the 'USER ID' Columns as it is unique identifier
dataset = dataset.drop("User ID",axis=1)
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [5]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [6]:
dataset['Purchased'].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [7]:
# Feature and target selection
independent = dataset[['Age', 'EstimatedSalary', 'Gender_Male']]
dependent = dataset['Purchased']

In [8]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [9]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [10]:
# Splitting data into training and test sets
x_train,x_test,y_train,y_test = train_test_split(independent,dependent,test_size=0.3,random_state=0)

In [11]:
x_train.shape

(280, 3)

In [12]:
x_test.shape

(120, 3)

In [13]:
y_train.shape

(280,)

In [14]:
y_test.shape

(120,)

In [15]:
# Data Standardization
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [16]:
# GridSearch CV
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Defining the different parameters for gridsearch optimization
param_grid = {'solver':[ 'lbfgs','newton-cg', 'liblinear', 'saga'],'penalty':['l2'],'C':[0.01,0.1,1,10]}
grid = GridSearchCV(LogisticRegression(),param_grid,refit=True,verbose=3,scoring='f1_weighted',n_jobs=-1)
grid.fit(x_train,y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10], 'penalty': ['l2'],
                         'solver': ['lbfgs', 'newton-cg', 'liblinear', 'saga']},
             scoring='f1_weighted', verbose=3)

In [17]:
# Best gridsearch parameter
print(grid.best_params_)

{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}


In [18]:
# Grid prediction
y_pred = grid.predict(x_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 1, 1])

In [19]:
# evaluation metrics
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[74  5]
 [10 31]]


In [20]:
from sklearn.metrics import classification_report
report = classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.88      0.94      0.91        79
           1       0.86      0.76      0.81        41

    accuracy                           0.88       120
   macro avg       0.87      0.85      0.86       120
weighted avg       0.87      0.88      0.87       120



In [21]:
from sklearn.metrics import f1_score
f1_weighted= f1_score(y_test,y_pred,average='weighted')
print(f1_weighted)

0.8728587363556688


In [22]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])

np.float64(0.9481321395492437)

In [23]:
results = grid.cv_results_
table = pd.DataFrame.from_dict(results)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.010828,0.005519,0.005945,0.001702,0.01,l2,lbfgs,"{'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}",0.755102,0.673469,0.618196,0.765625,0.681373,0.698753,0.054914,14
1,0.005676,0.001781,0.003759,0.000640,0.01,l2,newton-cg,"{'C': 0.01, 'penalty': 'l2', 'solver': 'newton...",0.755102,0.673469,0.618196,0.765625,0.681373,0.698753,0.054914,14
2,0.004080,0.001580,0.005413,0.002638,0.01,l2,liblinear,"{'C': 0.01, 'penalty': 'l2', 'solver': 'liblin...",0.795918,0.763089,0.616071,0.927778,0.926743,0.805920,0.116124,10
3,0.003067,0.000643,0.005517,0.001166,0.01,l2,saga,"{'C': 0.01, 'penalty': 'l2', 'solver': 'saga'}",0.755102,0.673469,0.618196,0.765625,0.681373,0.698753,0.054914,14
4,0.003699,0.000716,0.005163,0.002535,0.10,l2,lbfgs,"{'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}",0.812540,0.763089,0.644599,0.909115,0.888158,0.803500,0.095169,11
5,0.005068,0.001763,0.007266,0.002218,0.10,l2,newton-cg,"{'C': 0.1, 'penalty': 'l2', 'solver': 'newton-...",0.812540,0.763089,0.644599,0.909115,0.888158,0.803500,0.095169,11
6,0.002386,0.000869,0.004998,0.002934,0.10,l2,liblinear,"{'C': 0.1, 'penalty': 'l2', 'solver': 'libline...",0.819142,0.802399,0.644599,0.927778,0.926743,0.824132,0.103924,1
7,0.003240,0.001911,0.003856,0.000650,0.10,l2,saga,"{'C': 0.1, 'penalty': 'l2', 'solver': 'saga'}",0.812540,0.763089,0.644599,0.909115,0.888158,0.803500,0.095169,11
8,0.003123,0.000537,0.006016,0.004431,1.00,l2,lbfgs,"{'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}",0.835985,0.802399,0.644599,0.927778,0.890114,0.820175,0.097839,2
9,0.004034,0.000796,0.003436,0.000623,1.00,l2,newton-cg,"{'C': 1, 'penalty': 'l2', 'solver': 'newton-cg'}",0.835985,0.802399,0.644599,0.927778,0.890114,0.820175,0.097839,2


In [24]:
age = int(input("Enter your age:"))
salary = float(input("Enter your salary"))
gender = int(input("male=1,female=0"))

In [25]:
data = sc.transform([[age,salary,gender]])

C:\Users\thiru\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [26]:
result = grid.predict(data)
result

array([1])